# AutoShot Phase2: Shot + ClipShots Training

Attach datasets for source code, DatasetShot, ClipShots, and optionally a previous Kaggle output dataset for resume. Default settings are ready for Run All.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# Prefer Kaggle's built-in PyTorch on T4. P100 is sm_60 and may need a cu118 wheel.
gpu_name_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False,
)
gpu_name = gpu_name_result.stdout.strip().split('\n')[0] if gpu_name_result.returncode == 0 else ''
print('Kaggle GPU:', gpu_name or 'not detected')
if 'P100' in gpu_name:
    TORCH_INSTALL = [
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
        'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    ]
    print('P100 detected: installing PyTorch 2.4.1 + CUDA 11.8 for sm_60 ...')
    torch_install_result = subprocess.run(TORCH_INSTALL, check=False)
    if torch_install_result.returncode != 0:
        raise RuntimeError('Failed to install PyTorch cu118. Enable Kaggle Internet or switch Accelerator to T4.')
else:
    print('Non-P100 GPU detected: keeping Kaggle built-in PyTorch.')

import torch
print('torch version:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    print('torch arch list:', torch.cuda.get_arch_list())
    if torch.cuda.get_device_capability(0)[0] == 6 and 'sm_60' not in torch.cuda.get_arch_list():
        raise RuntimeError('P100 detected but installed PyTorch still lacks sm_60 support. Switch Accelerator to T4.')

print('Installing ffmpeg-python ...')
pip_result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ffmpeg-python'], check=False)
if pip_result.returncode != 0:
    print('pip install ffmpeg-python failed; notebook will use local fallback shim.')

SHOT_ROOT = Path('/kaggle/input/datasets/domanh704/shotdivide/DatasetShot')
CLIPSHOTS_ROOT = Path('/kaggle/input/datasets/domanh704/dataset-clipshots/ClipShots')
RUN_SMOKE_TEST = True
RUN_FULL_TRAIN = True

def find_dir_named(root, name):
    matches = [p for p in Path(root).rglob(name) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(f'Cannot find directory named {name} under {root}')
    return matches[0]

if not SHOT_ROOT.exists():
    SHOT_ROOT = find_dir_named('/kaggle/input', 'DatasetShot')
if not CLIPSHOTS_ROOT.exists():
    CLIPSHOTS_ROOT = find_dir_named('/kaggle/input', 'ClipShots')

source_candidates = [p.parent for p in Path('/kaggle/input').rglob('pyproject.toml')]
if not source_candidates:
    raise FileNotFoundError('Attach a source dataset containing pyproject.toml')
SOURCE_INPUT = sorted(source_candidates, key=lambda p: len(str(p)))[0]

previous_candidates = [
    p.parent for p in Path('/kaggle/input').rglob('phase2_shot_clipshots_resume.pt')
] + [
    p.parent for p in Path('/kaggle/input').rglob('shot_clipshots_phase2_sample_cache.pkl.partial.pkl')
] + [
    p.parent for p in Path('/kaggle/input').rglob('shot_clipshots_phase2_sample_cache.pkl')
]
PREVIOUS_OUTPUT = previous_candidates[0] if previous_candidates else None

WORK_DIR = Path('/kaggle/working/autoshotv2')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_INPUT, WORK_DIR)
if not (WORK_DIR / 'ckpt_0_200_0.pth').exists():
    ckpt_candidates = list(Path('/kaggle/input').rglob('ckpt_0_200_0.pth'))
    if not ckpt_candidates:
        raise FileNotFoundError('Attach ckpt_0_200_0.pth in source or a checkpoint input dataset')
    shutil.copy2(ckpt_candidates[0], WORK_DIR / 'ckpt_0_200_0.pth')

# Fallback shim only if ffmpeg-python is still unavailable.
try:
    import ffmpeg as _ffmpeg_test
    print('ffmpeg-python import OK')
except Exception:
    print('ffmpeg-python unavailable; writing local fallback ffmpeg.py shim')
    (WORK_DIR / 'ffmpeg.py').write_text(r'''
import subprocess

class _Input:
    def __init__(self, filename):
        self.filename = filename
        self.kwargs = {}

    def output(self, _pipe, **kwargs):
        self.kwargs = kwargs
        return self

    def run(self, capture_stdout=True, capture_stderr=True):
        size = self.kwargs.get('s') or self.kwargs.get('size') or '48x27'
        pix_fmt = self.kwargs.get('pix_fmt', 'rgb24')
        fmt = self.kwargs.get('format', 'rawvideo')
        cmd = ['ffmpeg', '-v', 'error', '-i', self.filename, '-vf', f'scale={size}', '-f', fmt, '-pix_fmt', pix_fmt, 'pipe:1']
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        return proc.stdout, proc.stderr

def input(filename):
    return _Input(filename)
''', encoding='utf-8')

if PREVIOUS_OUTPUT and PREVIOUS_OUTPUT.exists():
    for name in [
        'shot_clipshots_trainval.pickle',
        'shot_clipshots_phase2_sample_cache.pkl',
        'shot_clipshots_phase2_sample_cache.pkl.partial.pkl',
        'shot_clipshots_phase2_sample_cache.pkl.parts',
        'phase2_shot_clipshots_resume.pt',
        'phase2_shot_clipshots_checkpoints',
        'eval_cache_shot_clipshots',
    ]:
        src = PREVIOUS_OUTPUT / name
        dst = WORK_DIR / name
        if src.exists():
            if src.is_dir():
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)

os.chdir(WORK_DIR)
os.environ['PYTHONPATH'] = str(WORK_DIR / 'src')
print('SOURCE_INPUT     =', SOURCE_INPUT)
print('SHOT_ROOT        =', SHOT_ROOT)
print('CLIPSHOTS_ROOT   =', CLIPSHOTS_ROOT)
print('PREVIOUS_OUTPUT  =', PREVIOUS_OUTPUT)
print('WORK_DIR         =', WORK_DIR)

In [ ]:
!python -m autoshotv2.prepare_metadata \
  --shot-root "{SHOT_ROOT}" \
  --clipshots-root "{CLIPSHOTS_ROOT}" \
  --out /kaggle/working/autoshotv2/shot_clipshots_trainval.pickle \
  --val-ratio 0.10 \
  --max-val-videos 200

Optional smoke test. It is skipped by default so Run All goes straight to full training.

In [ ]:
import subprocess

if RUN_SMOKE_TEST:
    subprocess.run([
        'python', '-m', 'autoshotv2.train_phase2',
        '--meta', '/kaggle/working/autoshotv2/shot_clipshots_trainval.pickle',
        '--base-ckpt', '/kaggle/working/autoshotv2/ckpt_0_200_0.pth',
        '--epochs', '1',
        '--max-train-videos', '2',
        '--max-total-samples', '100',
        '--max-samples-per-video', '50',
        '--max-val-videos', '1',
        '--max-test-videos', '1',
        '--sample-cache', '/kaggle/working/autoshotv2/smoke_sample_cache.pkl',
        '--out-ckpt', '/kaggle/working/autoshotv2/smoke_ckpt.pth',
        '--results', '/kaggle/working/autoshotv2/smoke_results.pkl',
        '--eval-cache-dir', '/kaggle/working/autoshotv2/smoke_eval_cache',
        '--rebuild-sample-cache',
        '--no-eval-cache',
        '--no-resume',
    ], check=True)
else:
    print('RUN_SMOKE_TEST=False, skipping smoke test.')

Full training. `--stop-after-minutes 690` saves progress and exits before Kaggle's 12-hour limit.

In [ ]:
if RUN_FULL_TRAIN:
    subprocess.run([
        'python', '-m', 'autoshotv2.train_phase2',
        '--meta', '/kaggle/working/autoshotv2/shot_clipshots_trainval.pickle',
        '--base-ckpt', '/kaggle/working/autoshotv2/ckpt_0_200_0.pth',
        '--sample-cache', '/kaggle/working/autoshotv2/shot_clipshots_phase2_sample_cache.pkl',
        '--resume-state', '/kaggle/working/autoshotv2/phase2_shot_clipshots_resume.pt',
        '--checkpoint-dir', '/kaggle/working/autoshotv2/phase2_shot_clipshots_checkpoints',
        '--out-ckpt', '/kaggle/working/autoshotv2/ckpt_phase2_shot_clipshots_best.pth',
        '--results', '/kaggle/working/autoshotv2/phase2_shot_clipshots_results.pkl',
        '--eval-cache-dir', '/kaggle/working/autoshotv2/eval_cache_shot_clipshots',
        '--epochs', '20',
        '--batch-size', '512',
        '--max-samples-per-video', '160',
        '--save-every-videos', '25',
        '--save-every-epochs', '1',
        '--log-every-batches', '100',
        '--stop-after-minutes', '420',
    ], check=True)
else:
    print('RUN_FULL_TRAIN=False, skipping full training.')